# Diagnóstico de calidad de datos

**Proyecto:** Percepción de la belleza e influencia de las redes sociales
**Aprendiz:** Maileth Begambre · Tecnólogo en Análisis y Desarrollo de Software (ADSO)
**Evidencia:** GA1-220501094-01-AA1-EV06 – Diagnosticar la calidad de datos: preparación, exploración y documentación

Este cuaderno sigue la metodología de la Guía 4 (*Explorando Instacart*): **código → salida → interpretación**.
Las interpretaciones se escribieron a partir de las salidas reales de cada celda. No se usan datos de Instacart.

| Sección | Contenido | Momento EV06 |
|---|---|---|
| 1 | Comprensión del proyecto y de los datasets | 1 |
| 2 | Carga de los datos y diccionarios de códigos | 2 |
| 3 | Inspección de la estructura de cada dataset | 3 |
| 4 | Tabla de diagnóstico estructural | 3 |
| 5 | Diagnóstico de calidad (faltantes, tipos, inconsistencias, limitaciones, acciones) | 4 · *pendiente* |

## 1. Comprensión del proyecto y de los datasets

### Pregunta analítica (EV02)

> ¿Qué relación existe entre la frecuencia de exposición a contenido de belleza en redes sociales, la comparación de la apariencia con otras personas y los cambios que reportan los jóvenes de 18 a 25 años en su percepción de la belleza y sus preferencias de moda o maquillaje?

**Objetivo general.** Analizar, de forma descriptiva y exploratoria, la relación entre la exposición a contenido de belleza en redes sociales, la comparación de la apariencia y los cambios que reportan jóvenes de 18 a 25 años en su percepción de la belleza y sus preferencias de moda o maquillaje.

El proyecto separa tres dimensiones: **(1) exposición** (cuánto y qué contenido se ve), **(2) comparación** (si la persona compara su apariencia) y **(3) resultado percibido** (cambios en gustos, ideal de belleza o satisfacción). El análisis es descriptivo y exploratorio: no se afirmará causalidad.

### Datasets del proyecto

| DataFrame | Fuente | Formato | Qué contiene | Relación con el problema |
|---|---|---|---|---|
| `df_pritchard` | Pritchard (2021), Mendeley Data, CC BY 4.0 | SPSS `.sav` | Encuesta a estudiantes universitarios de EE. UU. sobre uso de Instagram, Twitter y Snapchat, comparación de apariencia e imagen corporal | Cubre exposición, comparación y satisfacción corporal. No mide moda ni maquillaje |
| `df_engeln` | Engeln et al. (2020), Mendeley Data, CC BY 4.0 | SPSS `.sav` | Experimento con mujeres universitarias de EE. UU.: 7 min de Instagram, Facebook o un juego de control; medidas antes y después | Cubre exposición controlada, comparación durante la tarea y satisfacción con la apariencia |
| `df_dane` | DANE, ENTIC Hogares 2024 (complementario, no se publica en GitHub) | CSV `;` | Encuesta oficial de uso de TIC en hogares de Colombia | Solo contexto colombiano de exposición: uso de internet y de redes sociales |

La **encuesta propia** es la fuente principal prevista, pero todavía no se ha aplicado; por eso no aparece en este cuaderno. Su diseño está en `data/diccionario_datos.md`.

### Aspectos de calidad que se deben revisar

- **Valores faltantes**, incluidos los que no aparecen como nulos (por ejemplo, textos vacíos) y los que se deben a preguntas que solo respondía una parte de las personas.
- **Tipos de datos:** las respuestas están guardadas como **códigos** (1, 2, 3…). Son categorías u órdenes, **no cantidades**: no se calcularán promedios de esos códigos sin justificarlo.
- **Rangos válidos:** edades dentro de 18 a 25 años y códigos que existan en el diccionario de etiquetas.
- **Identificadores y duplicados:** qué representa cada fila (persona, hogar, participante).
- **Cobertura:** población, país y variables que no existen en cada dataset.

### ¿Por qué explorar antes de analizar?

Porque un número en una tabla puede significar cosas distintas: una edad, un código de respuesta o un valor vacío. Si se analiza sin revisar la estructura, se pueden promediar categorías, contar filas como si fueran personas o ignorar respuestas faltantes, y las conclusiones serían incorrectas.

## 2. Preparación y carga de los datos

Cada dataset se carga **con el procedimiento que corresponde a su formato** y en un DataFrame independiente:

| Formato | Función | Por qué |
|---|---|---|
| SPSS `.sav` | `pd.read_spss(ruta, convert_categoricals=False)` | Conserva los **códigos originales** (1, 2, 3…) en lugar de reemplazarlos por texto |
| SPSS `.sav` (metadatos) | `pyreadstat.read_sav(ruta, metadataonly=True)` | Lee el diccionario que relaciona cada código con su etiqueta, sin cargar los datos otra vez |
| CSV del DANE | `pd.read_csv(ruta, sep=";", low_memory=False)` | El archivo usa punto y coma. `low_memory=False` hace que pandas revise la columna completa antes de decidir su tipo |

A diferencia de la Guía 4, aquí `sep=";"` solo aplica al CSV del DANE: los `.sav` no tienen separador.

In [1]:
from pathlib import Path
import re

import pandas as pd
import pyreadstat
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

# Funciona tanto si Jupyter se abre en la raíz del proyecto como en notebooks/
RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATOS = RAIZ / "data" / "raw"

RUTA_PRITCHARD = DATOS / "pritchard_2021_social_media_body_image.sav"
RUTA_ENGELN = DATOS / "engeln_2020_instagram_facebook.sav"
RUTA_DANE = DATOS / "dane" / "BASE DEFINITIVA ENTIC HOGARES 2024.csv"
RUTA_DANE_DICC = DATOS / "dane" / "DSO-ENTIC_H-DDT-001.xlsx"

for ruta in [RUTA_PRITCHARD, RUTA_ENGELN, RUTA_DANE, RUTA_DANE_DICC]:
    print("Encontrado " if ruta.exists() else "NO ENCONTRADO", ruta.relative_to(RAIZ))

Encontrado  data\raw\pritchard_2021_social_media_body_image.sav
Encontrado  data\raw\engeln_2020_instagram_facebook.sav
Encontrado  data\raw\dane\BASE DEFINITIVA ENTIC HOGARES 2024.csv
Encontrado  data\raw\dane\DSO-ENTIC_H-DDT-001.xlsx


In [2]:
df_pritchard = pd.read_spss(RUTA_PRITCHARD, convert_categoricals=False)
_, meta_pritchard = pyreadstat.read_sav(RUTA_PRITCHARD, metadataonly=True)

print("Pritchard cargado correctamente:", df_pritchard.shape)

Pritchard cargado correctamente: (404, 246)


In [3]:
df_engeln = pd.read_spss(RUTA_ENGELN, convert_categoricals=False)
_, meta_engeln = pyreadstat.read_sav(RUTA_ENGELN, metadataonly=True)

print("Engeln cargado correctamente:", df_engeln.shape)

Engeln cargado correctamente: (308, 70)


In [4]:
# El DANE no se publica en GitHub. Si falta, ver data/raw/README.md
if RUTA_DANE.exists():
    df_dane = pd.read_csv(RUTA_DANE, sep=";", low_memory=False)
    print("DANE cargado correctamente:", df_dane.shape)
else:
    df_dane = None
    print("No se encontró el CSV del DANE. Las celdas del DANE se omitirán.")
    print("Para descargarlo, sigue las instrucciones de data/raw/README.md")

DANE cargado correctamente: (148700, 377)


**Interpretación.** Los tres archivos se cargan sin errores, cada uno en su propio DataFrame. El CSV del DANE se lee con la codificación por defecto (UTF-8). Los números que muestran estas celdas son filas y columnas; en la sección 3 se revisa qué representa cada fila.

### 2.1 Diccionarios de códigos

Los archivos `.sav` guardan cada respuesta como un código y, aparte, la etiqueta de ese código (por ejemplo, `1 = Never`, `2 = Rarely`). Para no perder el significado, se construye un **diccionario de códigos** por dataset con cuatro columnas: variable, etiqueta de la variable (la pregunta), código y etiqueta del valor.

Los diccionarios de Pritchard y Engeln se guardan en `data/diccionarios/` para consultarlos sin abrir SPSS. El del DANE se arma desde su archivo oficial (`DSO-ENTIC_H-DDT-001.xlsx`) y no se exporta, porque el DANE no declara una licencia abierta.

In [5]:
def diccionario_codigos(meta):
    """Devuelve un DataFrame con variable, pregunta, código y etiqueta a partir de los metadatos SPSS."""
    filas = []
    for variable, pregunta in zip(meta.column_names, meta.column_labels):
        valores = meta.variable_value_labels.get(variable, {})
        if not valores:
            filas.append({"variable": variable, "etiqueta_variable": pregunta or "",
                          "codigo": pd.NA, "etiqueta_valor": pd.NA})
        for codigo, etiqueta in valores.items():
            filas.append({"variable": variable, "etiqueta_variable": pregunta or "",
                          "codigo": codigo, "etiqueta_valor": etiqueta})
    return pd.DataFrame(filas)


def etiquetas(diccionario, variable):
    """Muestra los códigos y etiquetas de una variable."""
    return diccionario.loc[diccionario["variable"] == variable, ["codigo", "etiqueta_valor"]]


dic_pritchard = diccionario_codigos(meta_pritchard)
dic_engeln = diccionario_codigos(meta_engeln)

CARPETA_DIC = RAIZ / "data" / "diccionarios"
CARPETA_DIC.mkdir(exist_ok=True)
dic_pritchard.to_csv(CARPETA_DIC / "pritchard_2021_codigos.csv", index=False)
dic_engeln.to_csv(CARPETA_DIC / "engeln_2020_codigos.csv", index=False)

for nombre, meta in [("Pritchard", meta_pritchard), ("Engeln", meta_engeln)]:
    sin_pregunta = sum(1 for etiqueta in meta.column_labels if not etiqueta)
    print(f"{nombre}: {len(meta.column_names)} variables, "
          f"{len(meta.variable_value_labels)} con etiquetas de valor, "
          f"{sin_pregunta} sin texto de pregunta")

Pritchard: 246 variables, 200 con etiquetas de valor, 21 sin texto de pregunta
Engeln: 70 variables, 56 con etiquetas de valor, 6 sin texto de pregunta


In [6]:
print("Pritchard · Q17:", meta_pritchard.column_names_to_labels["Q17"])
display(etiquetas(dic_pritchard, "Q17"))

print("Engeln · Condition:", meta_engeln.column_names_to_labels["Condition"])
display(etiquetas(dic_engeln, "Condition"))

Pritchard · Q17: What is the average amount of time you spend on Instagram each day?


,codigo,etiqueta_valor
14,1.0,0-15 min
15,2.0,15-30 min
16,3.0,1-2 hr
17,4.0,2-3 hr
18,5.0,3-4 hr
19,6.0,4-5 hr
20,7.0,5-6 hr
21,8.0,6-7 hr
22,9.0,7-8 hr
23,10.0,8-9 hr


Engeln · Condition: Condition


,codigo,etiqueta_valor
0,1.0,Browsed Facebook
1,2.0,Browsed Instagram
2,3.0,Played Bejeweled


**Interpretación.** El diccionario muestra que, por ejemplo, en Pritchard `Q17 = 3` no significa "3 horas" sino el intervalo **"1-2 hr"**, y que en Engeln `Condition = 2` significa **"Browsed Instagram"**. Por eso estos códigos se tratan como **categorías**, no como cantidades.

In [7]:
def diccionario_dane(ruta):
    """Arma el diccionario código → etiqueta a partir de la columna 'Dominios' del archivo oficial del DANE."""
    hoja = pd.read_excel(ruta, sheet_name="Diseño base de Datos", header=3)
    hoja.columns = [str(c).strip().split("\n")[0] for c in hoja.columns]
    filas = []
    for _, fila in hoja.dropna(subset=["Nombre de la variable o la columna"]).iterrows():
        variable = str(fila["Nombre de la variable o la columna"]).strip()
        pregunta = fila["Descripción de la variable o la columna"]
        dominio = str(fila["Dominios"]).replace("_x000D_", "\n")
        pares = re.findall(r"(\d+)\s*\.\s*([^\n]+)", dominio)
        if not pares:
            filas.append({"variable": variable, "etiqueta_variable": pregunta,
                          "codigo": pd.NA, "etiqueta_valor": pd.NA})
        for codigo, etiqueta in pares:
            filas.append({"variable": variable, "etiqueta_variable": pregunta,
                          "codigo": int(codigo), "etiqueta_valor": etiqueta.strip()})
    return pd.DataFrame(filas)


if RUTA_DANE_DICC.exists():
    dic_dane = diccionario_dane(RUTA_DANE_DICC)
    print("Variables en el diccionario del DANE:", dic_dane["variable"].nunique())
    print("IV12R13C1:", dic_dane.loc[dic_dane["variable"] == "IV12R13C1", "etiqueta_variable"].iloc[0])
    display(etiquetas(dic_dane, "IV12R13C1"))
else:
    dic_dane = None
    print("No se encontró el diccionario del DANE (ver data/raw/README.md).")

Variables en el diccionario del DANE: 377
IV12R13C1: ¿Utilizó las redes sociales para entretenimiento (ej. juego, farándula, seguir influenciadores)?


,codigo,etiqueta_valor
517,1,Sí
518,2,No


**Interpretación.** El archivo de datos del DANE no trae etiquetas: solo códigos de columna como `iv12r13c1`. El diccionario oficial permite saber que esa columna es la pregunta *"¿Utilizó las redes sociales para entretenimiento (ej. juego, farándula, seguir influenciadores)?"*, con `1 = Sí` y `2 = No`.

## 3. Inspección de la estructura de los datasets

Para cada DataFrame se ejecutan los comandos que pide la guía: `head()`, `tail()`, `shape`, `columns`, `dtypes`, `info()`, `describe()` y `describe(include="object")`.

**Advertencia para leer `describe()`:** calcula `mean` y `std` de todas las columnas numéricas, pero en estos datasets casi todas son **códigos de respuesta**. Esos promedios no tienen significado (el promedio de "Hombre = 1" y "Mujer = 2" no describe a nadie). De `describe()` se usan sobre todo `count` (cuántos valores no nulos hay), `min` y `max` (si los códigos están en el rango esperado).

### 3.1 Pritchard (2021) · `df_pritchard`

In [8]:
df_pritchard.head()

,age,gender,race,Q16,Q17,Q18,Q19,Q20,Q21,Q22,Q23,Q24,Q25,Q26,Q27,...,Eat_26,Dieting,bulimia,oralcontrol,filter_$,PC,PA,UBS,SA,WC,BodySurv,BodySham,Control,PACSR,GenderRE
0,18.0,2.0,5.0,1.0,3.0,3.0,2.0,,3.0,"Sometimes selfies, sometimes photos of me and my boyfrie...",As long as I don't look weird or am at an unflattering a...,2.0,,,2.0,...,5.0,5.0,0.0,0.0,1.0,22.0,39.0,20.0,42.0,28.0,3.375,1.625,3.500000,1.818182,0.0
1,17.0,2.0,3.0,1.0,2.0,2.0,1.0,I favor this time over others because it is most likely ...,2.0,I post pictures of myself with friends or with family. I...,"My first step would be to do my hair, my makeup, and put...",1.0,I use filters for the aesthetic of my feed.,I do consider how other users will respond to my images ...,2.0,...,2.0,1.0,0.0,1.0,1.0,32.0,43.0,29.0,48.0,38.0,3.000,2.000,3.500000,1.454545,0.0
2,18.0,2.0,5.0,1.0,3.0,4.0,1.0,I feel as though more people will be active.,3.0,I post selfies and pictures with my friends. I decide w...,I would send it to my friends to get input.,1.0,I feel like it makes me look better,I do because if I dont think I will get a lot of likes o...,1.0,...,29.0,22.0,2.0,5.0,1.0,30.0,27.0,26.0,32.0,28.0,4.375,3.250,3.714286,3.272727,0.0
3,18.0,1.0,5.0,1.0,2.0,3.0,1.0,Golden Hour,2.0,Pictures with friends or girlfriends. Ones that look goo...,Golden hour and non-blurry,2.0,,,2.0,...,5.0,1.0,1.0,3.0,0.0,37.0,35.0,28.0,44.0,31.0,3.000,1.875,3.625000,2.272727,1.0
4,19.0,2.0,5.0,1.0,1.0,2.0,2.0,,2.0,"I never post images of just myself, there is always at l...","I would make sure I have makeup on, and try to be in a p...",1.0,I edit and use filters to hide blemishes and make myself...,"Yes, I always make sure the picture is appropriate and t...",2.0,...,0.0,0.0,0.0,0.0,1.0,28.0,32.0,23.0,38.0,28.0,3.000,3.000,3.250000,2.545455,0.0


**Interpretación.** Cada fila es una persona que respondió la encuesta. Las primeras columnas son `age`, `gender` y `race`, seguidas de preguntas con nombres en código (`Q16`, `Q17`…). Los valores aparecen como `18.0`, `2.0`, `5.0`: son números con decimal aunque representen edades o códigos. En la columna de texto `Q20` la primera fila aparece **vacía**.

In [9]:
df_pritchard.tail()

,age,gender,race,Q16,Q17,Q18,Q19,Q20,Q21,Q22,Q23,Q24,Q25,Q26,Q27,...,Eat_26,Dieting,bulimia,oralcontrol,filter_$,PC,PA,UBS,SA,WC,BodySurv,BodySham,Control,PACSR,GenderRE
399,18.0,2.0,3.0,1.0,6.0,4.0,1.0,Most people go on instagram at night so that is when you...,4.0,Selfies or pictures taken of me. I decide by asking peop...,If it is relevant enough so that people will want to see...,1.0,To make it look better or to enhance the picture,"Yes, because that was the main purpose of posting the pi...",1.0,...,29.0,22.0,5.0,2.0,1.0,29.0,35.0,28.0,40.0,31.0,3.625,2.750,3.000000,1.454545,0.0
400,22.0,1.0,5.0,1.0,3.0,3.0,1.0,There's more activity on Instagram in the evening. Every...,2.0,I post photos of myself with friends or doing activities...,Make sure I look nice and am dressed appropriately.,2.0,,,2.0,...,6.0,6.0,0.0,0.0,0.0,43.0,45.0,36.0,58.0,37.0,3.500,2.625,3.714286,0.636364,1.0
401,18.0,2.0,5.0,1.0,3.0,4.0,1.0,Its relaxing,3.0,I mostly post selfies or pictures of activities im doing.,I would have to put on makeup and do my hair.,1.0,For better quality pictures,no I don't consider how others will respond.,2.0,...,3.0,3.0,0.0,0.0,1.0,27.0,33.0,27.0,41.0,31.0,3.000,NaN,5.000000,0.000000,0.0
402,18.0,1.0,1.0,1.0,2.0,5.0,1.0,"If pictures are posted at certain times of the day, more...",3.0,Used to post selfies but I feel like that's more for Sna...,Probably set the phone down or put the phone at an angle...,1.0,If i edit my pictures depending on what I want to get ac...,"Yes. Just because at this point, when I post a picture o...",2.0,...,3.0,0.0,0.0,3.0,0.0,29.0,34.0,18.0,41.0,27.0,3.500,NaN,5.000000,0.454545,1.0
403,18.0,1.0,5.0,1.0,3.0,5.0,2.0,,2.0,I don't really post a type. If I decide that it can be p...,"Would my friends, parents, others approve? Does it portr...",1.0,It enhances the photo to make it look better.,"Yes, I think that others will judge you by the photos yo...",2.0,...,5.0,5.0,0.0,0.0,0.0,35.0,35.0,28.0,42.0,29.0,3.125,2.875,3.250000,2.272727,1.0


**Interpretación.** Las últimas filas tienen la misma estructura; la numeración del índice termina en 403, porque empieza en 0. También aparecen celdas de texto vacías en las respuestas abiertas.

In [10]:
df_pritchard.shape

(404, 246)

**Interpretación.** El DataFrame tiene **404 filas y 246 columnas**. 404 es la cantidad de **registros**, no necesariamente de personas válidas para el proyecto: falta comprobar edades, duplicados y respuestas incompletas (sección 5).

In [11]:
print(df_pritchard.columns.tolist())

['age', 'gender', 'race', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26', 'Q27', 'Q28', 'Q29', 'Q30', 'Q13_1', 'Q13_2', 'Q13_3', 'Q13_4', 'Q13_5', 'Q13_6', 'Q13_7', 'Q13_8', 'Q13_9', 'Q13_10', 'Q13_11', 'Q13_12', 'Q13_13', 'Q13_14', 'Q13_15', 'Q23_1', 'Q23_2', 'Q23_3', 'Q23_4', 'Q23_5', 'Q23_6', 'Q23_7', 'Q15_1', 'Q15_2', 'Q15_3', 'Q15_4', 'Q15_5', 'Q15_6', 'Q15_7', 'Q15_8', 'Q15_9', 'Q15_10', 'Q15_11', 'Q15_12', 'Q15_13', 'Q15_14', 'Q15_15', 'Q15_16', 'Q15_17', 'Q15_18', 'Q15_19', 'Q15_20', 'Q15_21', 'Q15_22', 'Q15_1.0', 'Q15_2.0', 'Q15_3.0', 'Q15_4.0', 'Q15_5.0', 'Q15_6.0', 'Q15_7.0', 'Q15_8.0', 'Q15_9.0', 'Q15_10.0', 'Q15_11.0', 'Q15_12.0', 'Q15_13.0', 'Q15_14.0', 'Q15_15.0', 'Q15_16.0', 'Q15_17.0', 'Q15_18.0', 'Q15_19.0', 'Q15_20.0', 'Q15_21.0', 'Q15_22.0', 'Q15_23', 'Q15_24', 'Q15_25', 'Q15_26', 'Q17_1', 'Q17_2', 'Q17_3', 'Q17_4', 'Q19_1', 'Q19_2', 'Q32', 'Q34', 'Q35', 'Q33', 'Q36', 'Q38', 'Q39', 'Q40', 'Q41', 'Q42', 'Q43', 'Q44', 'Q45', 'Q46', 'Q47', 

**Interpretación.** Los nombres son códigos del formulario original (`Q16`, `Q65_1`…), así que no se entienden sin el diccionario. Hay detalles que pueden confundir:
- La numeración no sigue el orden del cuestionario (`Q13_…` aparece después de `Q30`).
- `Q23` es una pregunta abierta y `Q23_1` … `Q23_7` son otra escala distinta.
- Hay nombres como `Q15_1` y `Q15_1.0`, que son preguntas diferentes.
- Al final hay puntajes ya calculados (`DFM`, `PACSR`, `BodySurv`…) y una variable técnica de SPSS (`filter_$`).

In [12]:
df_pritchard.dtypes

age         float64
gender      float64
race        float64
Q16         float64
Q17         float64
             ...   
BodySurv    float64
BodySham    float64
Control     float64
PACSR       float64
GenderRE    float64
Length: 246, dtype: object

In [13]:
df_pritchard.dtypes.value_counts()

float64    221
str         25
Name: count, dtype: int64

**Interpretación.** Hay **221 columnas `float64`** (números con decimal) y **25 columnas de texto** (`str`; en pandas 3 el texto ya no aparece como `object`). Las respuestas cerradas (`gender`, `Q17`…) están guardadas como `float64`, pero son **códigos de categorías**. El tipo no indica que sean cantidades. La única variable claramente cuantitativa al inicio es `age`.

In [14]:
df_pritchard.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 246 columns):
 #    Column          Non-Null Count  Dtype  
---   ------          --------------  -----  
 0    age             403 non-null    float64
 1    gender          403 non-null    float64
 2    race            403 non-null    float64
 3    Q16             404 non-null    float64
 4    Q17             385 non-null    float64
 5    Q18             371 non-null    float64
 6    Q19             384 non-null    float64
 7    Q20             404 non-null    str    
 8    Q21             385 non-null    float64
 9    Q22             404 non-null    str    
 10   Q23             404 non-null    str    
 11   Q24             361 non-null    float64
 12   Q25             404 non-null    str    
 13   Q26             404 non-null    str    
 14   Q27             362 non-null    float64
 15   Q28             404 non-null    str    
 16   Q29             404 non-null    str    
 17   Q30             404 non-n

**Interpretación.** `info()` muestra, para cada columna, cuántos valores **no nulos** tiene. Muchas columnas tienen menos de 404; las que menos tienen son `Q41` y `Q44` (54 valores), que pertenecen al bloque de preguntas sobre **Twitter**. Una explicación probable es que esas preguntas solo las respondían quienes usan Twitter; se verificará en la sección 5.

Las columnas de texto aparecen con **404 no nulos**, pero en `head()` ya se vieron celdas vacías. Eso significa que las respuestas en blanco están guardadas como texto vacío (`""`) y **no se cuentan como faltantes**.

In [15]:
df_pritchard.describe()

,age,gender,race,Q16,Q17,Q18,Q19,Q21,Q24,Q27,Q13_1,Q13_2,Q13_3,Q13_4,Q13_5,...,Eat_26,Dieting,bulimia,oralcontrol,filter_$,PC,PA,UBS,SA,WC,BodySurv,BodySham,Control,PACSR,GenderRE
count,403.000000,403.000000,403.000000,404.000000,385.000000,371.000000,384.000000,385.000000,361.000000,362.000000,402.000000,403.000000,402.000000,402.000000,403.000000,...,402.000000,402.000000,402.000000,402.000000,403.000000,403.000000,403.000000,403.000000,403.000000,403.000000,391.000000,359.000000,389.000000,395.000000,398.000000
mean,18.627792,1.615385,4.838710,1.047030,3.023377,4.013477,1.552083,2.636364,1.318560,1.657459,4.315920,3.297767,2.385572,2.159204,1.950372,...,10.544776,6.671642,1.405473,2.467662,0.583127,31.069479,37.506203,26.560794,45.168734,30.933002,3.347278,2.654596,3.707951,1.785316,0.409548
std,1.367869,0.540372,0.881705,0.211965,1.366250,1.530706,0.497929,0.772393,0.466564,0.475216,1.361089,1.614816,1.605194,1.543881,1.390271,...,12.038352,7.773389,2.898494,3.292885,0.493654,7.856938,7.630903,6.569805,8.674106,9.410113,0.756784,0.843703,0.637097,1.209592,0.492369
min,17.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,11.000000,8.000000,13.000000,10.000000,1.000000,1.000000,1.250000,0.000000,0.000000
25%,18.000000,1.000000,5.000000,1.000000,2.000000,3.000000,1.000000,2.000000,1.000000,1.000000,3.000000,2.000000,1.000000,1.000000,1.000000,...,3.000000,1.250000,0.000000,0.000000,0.000000,26.000000,33.000000,23.000000,39.500000,25.000000,2.875000,2.000000,3.250000,0.909091,0.000000
50%,18.000000,2.000000,5.000000,1.000000,3.000000,4.000000,2.000000,3.000000,1.000000,2.000000,4.000000,3.000000,2.000000,1.000000,1.000000,...,6.000000,4.000000,0.000000,2.000000,1.000000,32.000000,36.000000,26.000000,45.000000,31.000000,3.375000,2.625000,3.750000,1.818182,0.000000
75%,19.000000,2.000000,5.000000,1.000000,3.000000,5.000000,2.000000,3.000000,2.000000,2.000000,5.750000,5.000000,3.000000,3.000000,2.000000,...,13.000000,9.000000,1.000000,3.000000,1.000000,37.000000,43.000000,31.000000,51.000000,38.000000,3.875000,3.250000,4.125000,2.727273,1.000000
max,26.000000,4.000000,7.000000,2.000000,12.000000,7.000000,2.000000,4.000000,2.000000,2.000000,6.000000,6.000000,6.000000,6.000000,6.000000,...,72.000000,39.000000,18.000000,21.000000,1.000000,45.000000,55.000000,40.000000,65.000000,50.000000,4.875000,5.000000,5.000000,4.000000,1.000000


**Interpretación.**
- `age`: `count` = 403, así que falta una edad. `min` = 17 y `max` = 26: hay edades **fuera del rango de 18 a 25** del proyecto.
- `gender`: va de 1 a 4. Según el diccionario, son categorías (Male, Female, Transgender, Other); su `mean` no se interpreta.
- `Q17` (tiempo diario en Instagram): va de 1 a 12. Son **intervalos de tiempo codificados**, no horas.
- Las variables tienen `count` distintos, lo que confirma que la cantidad de respuestas cambia de una pregunta a otra.

In [16]:
df_pritchard.describe(include="object")

C:\Users\maile\AppData\Local\Temp\ipykernel_27604\3351535375.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_pritchard.describe(include="object")


,Q20,Q22,Q23,Q25,Q26,Q28,Q29,Q30,Q36,Q39,Q40,Q42,Q43,Q45,Q46,Q47,Q53,Q55,Q56,Q58,Q59,Q61,Q62,Q63,id
count,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404
unique,170,361,354,243,244,78,121,381,7,55,55,25,25,6,8,173,16,283,277,147,146,22,34,329,403
top,,,,,,,,,,,,,,,,,,,,,,,,,10720
freq,234,44,46,159,159,282,282,24,398,350,350,380,380,397,397,223,389,120,121,255,255,367,367,62,2


**Interpretación.** Las 25 columnas de texto tienen `count` = 404, pero el valor más frecuente (`top`) que aparece en blanco es el **texto vacío**. Por ejemplo, en `Q20` el texto vacío se repite 234 veces (`freq`). Esto confirma que buena parte de las respuestas abiertas no se contestó, aunque pandas no lo cuente como nulo.

**Sobre el aviso `Pandas4Warning`.** En pandas 3 el texto tiene su propio tipo (`str`). `include="object"` todavía incluye esas columnas por compatibilidad, y por eso el resultado es correcto, pero pandas avisa que en una versión futura dejará de hacerlo. Se conserva el comando porque es el que pide la guía; en pandas 4 habría que escribir `describe(include="str")`.

### 3.2 Engeln et al. (2020) · `df_engeln`

In [17]:
df_engeln.head()

,Condition,NA1,PA1,NA2,PA2,NA3,PA3,NA4,PA4,PA5,NA5,NA6,PA6,NA7,PA7,...,NegAffectPost,PosAffectPost,BISSTotalPre,BISSTotalPost,imagepeople,imageobject,reading,videopeople,videoobject,other,appearancethoughts,overallappcompduringtask,specbodypartcompduringtask,comparisonsduringtask,Age
0,1.0,2.0,4.0,4.0,3.0,2.0,3.0,3.0,5.0,4.0,1.0,1.0,3.0,1.0,4.0,...,1.9,3.9,7.166667,7.333333,0.0,15.0,25.0,0.0,60.0,0.0,1.0,1.0,1.0,1.0,19.0
1,1.0,1.0,1.0,3.0,1.0,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,2.0,NaN,...,2.6,1.6,3.666667,3.833333,0.0,50.0,20.0,30.0,0.0,0.0,1.0,1.0,1.0,1.0,19.0
2,1.0,1.0,3.0,1.0,4.0,1.0,3.0,3.0,4.0,4.0,1.0,2.0,4.0,1.0,3.0,...,1.7,2.8,6.666667,4.166667,0.0,45.0,5.0,50.0,0.0,0.0,1.0,1.0,1.0,1.0,19.0
3,1.0,1.0,3.0,1.0,4.0,1.0,1.0,1.0,3.0,2.0,1.0,1.0,3.0,2.0,4.0,...,1.4,2.8,5.833333,5.500000,5.0,50.0,35.0,5.0,5.0,0.0,1.0,1.0,1.0,1.0,18.0
4,1.0,1.0,1.0,4.0,3.0,1.0,3.0,2.0,3.0,3.0,1.0,1.0,4.0,3.0,4.0,...,1.2,3.2,6.000000,6.000000,0.0,45.0,0.0,5.0,50.0,0.0,2.0,1.0,1.0,1.0,20.0


**Interpretación.** Cada fila es una participante del experimento. La primera columna, `Condition`, indica el grupo (en las primeras filas es 1, *Browsed Facebook*). Siguen ítems de emociones con nombres abreviados (`NA1` = emoción negativa 1, `PA1` = emoción positiva 1), todos como números con decimal.

In [18]:
df_engeln.tail()

,Condition,NA1,PA1,NA2,PA2,NA3,PA3,NA4,PA4,PA5,NA5,NA6,PA6,NA7,PA7,...,NegAffectPost,PosAffectPost,BISSTotalPre,BISSTotalPost,imagepeople,imageobject,reading,videopeople,videoobject,other,appearancethoughts,overallappcompduringtask,specbodypartcompduringtask,comparisonsduringtask,Age
303,3.0,1.0,2.0,1.0,2.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.1,2.1,3.000000,3.666667,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,18.0
304,3.0,1.0,5.0,1.0,5.0,1.0,3.0,3.0,3.0,3.0,1.0,1.0,4.0,1.0,4.0,...,1.3,3.9,5.166667,5.166667,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,20.0
305,3.0,1.0,2.0,1.0,3.0,1.0,3.0,2.0,2.0,2.0,1.0,1.0,2.0,3.0,2.0,...,1.8,2.6,4.166667,4.166667,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,18.0
306,3.0,1.0,2.0,3.0,4.0,1.0,1.0,2.0,2.0,1.0,1.0,1.0,3.0,3.0,2.0,...,2.4,2.7,4.333333,4.500000,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,19.0
307,3.0,1.0,3.0,1.0,3.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.1,2.5,6.166667,7.000000,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,19.0


**Interpretación.** Las últimas filas pertenecen a `Condition` = 3 (*Played Bejeweled*, el grupo de control). El archivo parece estar **ordenado por grupo**.

In [19]:
df_engeln.shape

(308, 70)

**Interpretación.** Tiene **308 filas y 70 columnas**. Es mucho más pequeño que Pritchard y está centrado en un experimento puntual.

In [20]:
print(df_engeln.columns.tolist())

['Condition', 'NA1', 'PA1', 'NA2', 'PA2', 'NA3', 'PA3', 'NA4', 'PA4', 'PA5', 'NA5', 'NA6', 'PA6', 'NA7', 'PA7', 'NA8', 'PA8', 'PA9', 'NA9', 'PA10', 'NA10', 'BISS1pre', 'BISS2pre', 'BISS3pre', 'BISS4pre', 'BISS5pre', 'BISS6pre', 'NA1post', 'PA1post', 'NA2post', 'PA2post', 'NA3post', 'PA3post', 'NA4post', 'PA4post', 'PA5post', 'NA5post', 'NA6post', 'PA6post', 'NA7post', 'PA7post', 'NA8post', 'PA8post', 'PA9post', 'NA9post', 'PA10post', 'NA10post', 'BISS1post', 'BISS2post', 'BISS3post', 'BISS4post', 'BISS5post', 'BISS6post', 'NegAffectPre', 'PosAffectPre', 'NegAffectPost', 'PosAffectPost', 'BISSTotalPre', 'BISSTotalPost', 'imagepeople', 'imageobject', 'reading', 'videopeople', 'videoobject', 'other', 'appearancethoughts', 'overallappcompduringtask', 'specbodypartcompduringtask', 'comparisonsduringtask', 'Age']


**Interpretación.** Las columnas siguen el orden del experimento: emociones **antes** (`NA1`…`PA10`), satisfacción corporal antes (`BISS1pre`…), emociones y satisfacción **después** (`…post`), puntajes totales (`BISSTotalPre`, `BISSTotalPost`), porcentaje de tiempo mirando cada tipo de contenido (`imagepeople`…), comparación durante la tarea y `Age`. No hay variable de género porque todas las participantes eran mujeres.

In [21]:
df_engeln.dtypes

Condition                     float64
NA1                           float64
PA1                           float64
NA2                           float64
PA2                           float64
                               ...   
appearancethoughts            float64
overallappcompduringtask      float64
specbodypartcompduringtask    float64
comparisonsduringtask         float64
Age                           float64
Length: 70, dtype: object

In [22]:
df_engeln.dtypes.value_counts()

float64    70
Name: count, dtype: int64

**Interpretación.** Las **70 columnas son `float64`**. `Condition` es una categoría (1, 2, 3) aunque esté guardada como número. Los ítems de emociones y de comparación son escalas ordinales. Algunas columnas sí son cantidades: `Age`, los porcentajes de tiempo y los puntajes totales.

In [23]:
df_engeln.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 308 entries, 0 to 307
Data columns (total 70 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Condition                   308 non-null    float64
 1   NA1                         308 non-null    float64
 2   PA1                         308 non-null    float64
 3   NA2                         307 non-null    float64
 4   PA2                         308 non-null    float64
 5   NA3                         308 non-null    float64
 6   PA3                         308 non-null    float64
 7   NA4                         308 non-null    float64
 8   PA4                         308 non-null    float64
 9   PA5                         308 non-null    float64
 10  NA5                         308 non-null    float64
 11  NA6                         308 non-null    float64
 12  PA6                         307 non-null    float64
 13  NA7                         308 non-null    fl

**Interpretación.** La mayoría de columnas tiene 308 valores, pero las variables de la tarea (`imagepeople`, `reading`, `overallappcompduringtask`, `comparisonsduringtask`…) tienen **201 o 202**. Esas preguntas se refieren a lo que se vio en Instagram o Facebook, así que es probable que el grupo de control no las respondiera; se verificará en la sección 5. `BISSTotalPre` tiene 300, por lo que faltan algunos puntajes iniciales.

In [24]:
df_engeln.describe()

,Condition,NA1,PA1,NA2,PA2,NA3,PA3,NA4,PA4,PA5,NA5,NA6,PA6,NA7,PA7,...,NegAffectPost,PosAffectPost,BISSTotalPre,BISSTotalPost,imagepeople,imageobject,reading,videopeople,videoobject,other,appearancethoughts,overallappcompduringtask,specbodypartcompduringtask,comparisonsduringtask,Age
count,308.000000,308.000000,308.000000,307.000000,308.000000,308.000000,308.000000,308.000000,308.000000,308.000000,308.000000,308.000000,307.000000,308.000000,307.000000,...,308.000000,308.000000,300.000000,308.000000,202.000000,201.000000,202.000000,202.000000,201.000000,201.000000,308.000000,202.000000,202.000000,202.000000,308.000000
mean,2.022727,1.269481,2.418831,1.869707,3.084416,1.233766,1.970779,1.840909,2.873377,2.461039,1.279221,1.292208,2.364821,1.970779,2.716612,...,1.595779,2.488961,4.517778,4.484307,36.410891,26.706468,11.393564,12.094059,8.457711,5.136816,2.487013,3.331683,3.009901,3.170792,19.396104
std,0.816844,0.667127,1.093318,1.085839,0.923464,0.663543,1.015735,1.048039,0.909705,1.028158,0.712812,0.773547,1.077306,1.136795,1.120728,...,0.583666,0.849377,1.351829,1.337543,27.679482,22.474839,13.301136,16.994763,15.651819,13.799364,1.649151,1.804882,1.884712,1.705067,1.231608
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.100000,1.100000,0.833333,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,18.000000
25%,1.000000,1.000000,2.000000,1.000000,3.000000,1.000000,1.000000,1.000000,2.000000,2.000000,1.000000,1.000000,1.000000,1.000000,2.000000,...,1.100000,1.800000,3.500000,3.666667,10.000000,10.000000,2.000000,0.000000,0.000000,0.000000,1.000000,2.000000,1.000000,1.500000,18.000000
50%,2.000000,1.000000,2.000000,2.000000,3.000000,1.000000,2.000000,1.500000,3.000000,2.000000,1.000000,1.000000,2.000000,2.000000,3.000000,...,1.400000,2.400000,4.333333,4.166667,30.000000,20.000000,10.000000,5.000000,0.000000,0.000000,2.000000,3.000000,2.000000,3.000000,19.000000
75%,3.000000,1.000000,3.000000,2.000000,4.000000,1.000000,3.000000,2.000000,3.000000,3.000000,1.000000,1.000000,3.000000,3.000000,3.500000,...,1.800000,3.000000,5.500000,5.500000,60.000000,40.000000,15.000000,18.750000,10.000000,3.000000,4.000000,5.000000,5.000000,4.500000,20.000000
max,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,...,5.000000,5.000000,8.000000,8.500000,90.000000,95.000000,85.000000,80.000000,100.000000,90.000000,7.000000,7.000000,7.000000,7.000000,26.000000


**Interpretación.**
- `Condition`: va de 1 a 3, coherente con los tres grupos del diccionario.
- `Age`: va de 18 a **26**, así que hay al menos una edad fuera del rango del proyecto.
- `comparisonsduringtask`: va de 1 a 7, como su escala de "no comparison" a "a lot of comparison"; `count` = 202 confirma los faltantes de la tarea.
- Los `mean` de ítems ordinales (`NA1`, `PA1`…) no se interpretan como cantidades.

In [25]:
columnas_texto = df_engeln.select_dtypes(include="object").columns
if len(columnas_texto) > 0:
    display(df_engeln.describe(include="object"))
else:
    print("df_engeln no tiene columnas de texto: describe(include='object') no aplica.")

df_engeln no tiene columnas de texto: describe(include='object') no aplica.


**Interpretación.** Engeln no tiene columnas de texto. Si se ejecuta `describe(include="object")` directamente, pandas da un error `ValueError`, porque no encuentra columnas de ese tipo. Por eso la celda primero comprueba si existen, como indica la guía: *"cuando existan"*.

### 3.3 DANE ENTIC Hogares 2024 · `df_dane`

Las celdas de esta sección solo se ejecutan si el CSV del DANE está en `data/raw/dane/`.

In [26]:
if df_dane is not None:
    display(df_dane.head())

,ID,UID_ENCUESTA,uid_padre,UUID,UUID_1,p1r1c1,p2r1c1,p3r1c1,FEX_c,ii1r1c1,ii1r1c2,ii2r1c1,ii3r1c1,ii4r1c1,ii5r1c1,...,v6r2c1,v6r3c1,v6r4c1,v6r5c1,v6r6c1,v6r7c1,v6r8c1,v6r9c1,v6r10c1,v7r1c1,v7r2c1,v7r3c1,v7r4c1,v7r5c1,v7r6c1
0,59694,a0dc1da3-ce83-4d87-860e-d451da72dbc4,a0dc1da3-ce83-4d87-860e-d451da72dbc4,a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1],a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1]/Personas[1],15,15001,1,"149,227493",1,3.0,1,3,3,2,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
1,59694,a0dc1da3-ce83-4d87-860e-d451da72dbc4,a0dc1da3-ce83-4d87-860e-d451da72dbc4,a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1],a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1]/Personas[2],15,15001,1,"149,227493",1,3.0,1,3,3,2,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
2,59694,a0dc1da3-ce83-4d87-860e-d451da72dbc4,a0dc1da3-ce83-4d87-860e-d451da72dbc4,a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1],a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1]/Personas[3],15,15001,1,"149,227493",1,3.0,1,3,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,59695,7a8612ad-6679-402f-9800-1b31be670a17,7a8612ad-6679-402f-9800-1b31be670a17,7a8612ad-6679-402f-9800-1b31be670a17/Hogares[1],7a8612ad-6679-402f-9800-1b31be670a17/Hogares[1]/Personas[1],15,15001,1,"231,7540904",1,3.0,1,4,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,59695,7a8612ad-6679-402f-9800-1b31be670a17,7a8612ad-6679-402f-9800-1b31be670a17,7a8612ad-6679-402f-9800-1b31be670a17/Hogares[1],7a8612ad-6679-402f-9800-1b31be670a17/Hogares[1]/Personas[2],15,15001,1,"231,7540904",1,3.0,1,4,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Interpretación.** Las primeras columnas son identificadores (`ID`, `UID_ENCUESTA`, `uid_padre`, `UUID`…), luego `p1r1c1` (departamento), `p2r1c1` (municipio) y `FEX_c` (factor de expansión). Las preguntas tienen nombres en código (`ii1r1c1`…). `FEX_c` aparece como `149,227493`: un número escrito **con coma decimal**.

In [27]:
if df_dane is not None:
    display(df_dane.tail())

,ID,UID_ENCUESTA,uid_padre,UUID,UUID_1,p1r1c1,p2r1c1,p3r1c1,FEX_c,ii1r1c1,ii1r1c2,ii2r1c1,ii3r1c1,ii4r1c1,ii5r1c1,...,v6r2c1,v6r3c1,v6r4c1,v6r5c1,v6r6c1,v6r7c1,v6r8c1,v6r9c1,v6r10c1,v7r1c1,v7r2c1,v7r3c1,v7r4c1,v7r5c1,v7r6c1
148695,119603,e465e8a7-bccd-42cd-a9ea-5562806fbcc0,e465e8a7-bccd-42cd-a9ea-5562806fbcc0,e465e8a7-bccd-42cd-a9ea-5562806fbcc0/Hogares[1],e465e8a7-bccd-42cd-a9ea-5562806fbcc0/Hogares[1]/Personas[1],76,76001,1,"1490,035081",1,2.0,1,1,2,2,...,2.0,3.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,1.0
148696,119604,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1],cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1]/Personas[1],76,76001,1,"2152,610599",1,1.0,1,4,3,1,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
148697,119604,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1],cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1]/Personas[2],76,76001,1,"2152,610599",1,1.0,1,4,3,1,...,3.0,3.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
148698,119604,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1],cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1]/Personas[3],76,76001,1,"2152,610599",1,1.0,1,4,3,1,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
148699,119604,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1],cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1]/Personas[4],76,76001,1,"2152,610599",1,1.0,1,4,3,1,...,2.0,2.0,2.0,2.0,2.0,2.0,1.0,1.0,2.0,1.0,2.0,2.0,1.0,NaN,NaN


**Interpretación.** Las últimas filas mantienen la estructura. Muchas columnas aparecen como `NaN`, lo que anticipa una gran cantidad de valores vacíos.

In [28]:
if df_dane is not None:
    print(df_dane.shape)
    print("Encuestas (UID_ENCUESTA) distintas:", df_dane["UID_ENCUESTA"].nunique())

(148700, 377)
Encuestas (UID_ENCUESTA) distintas: 53343


**Interpretación.** El archivo tiene **148.700 filas y 377 columnas**, pero solo **53.343 valores distintos de `UID_ENCUESTA`**. Igual que en la Guía 4, donde las filas no eran clientes, aquí **cada fila es una persona** y varias personas comparten la misma encuesta de hogar. El número de filas no es el número de hogares.

In [29]:
if df_dane is not None:
    print(df_dane.columns.tolist())

['ID', 'UID_ENCUESTA', 'uid_padre', 'UUID', 'UUID_1', 'p1r1c1', 'p2r1c1', 'p3r1c1', 'FEX_c', 'ii1r1c1', 'ii1r1c2', 'ii2r1c1', 'ii3r1c1', 'ii4r1c1', 'ii5r1c1', 'ii7r1c1', 'ii6r1c2', 'ii7r2c1', 'ii6r2c2', 'ii7r3c1', 'ii6r3c2', 'ii7r4c1', 'ii6r4c2', 'ii7r5c1', 'ii6r5c2', 'ii7r6c1', 'ii6r6c2', 'ii9r1c1', 'ii9r2c1', 'ii9r3c1', 'ii9r4c1', 'ii10r1c1', 'ii10r1c2', 'ii10r1c3', 'ii11r1c1', 'ii12r1c1', 'ii13r1c1', 'ii14r1c1', 'ii15ar1c1', 'ii15r1c1', 'ii17r1c1', 'ii18r1c1', 'ii18r1c2', 'ii18r1c3', 'ii18r2c1', 'ii18r3c1', 'ii18r4c1', 'ii8r1c1', 'ii16r1c1', 'ii19r1c1', 'ii19r2c1', 'ii19r3c1', 'ii19r4c1', 'ii19r5c1', 'iii2r1c1', 'iii4r1c1', 'iii5r1c1', 'iii6r1c1', 'iii7r1c1', 'iii8r1c1', 'iii10r1c1', 'iii11r1c1', 'iii11r2c1', 'iii11r3c1', 'iii11r4c1', 'iii11r5c1', 'iii11r6c1', 'iii11r7c1', 'iii11r8c1', 'iii12r1c1', 'iii13r1c1', 'iii14r1c1', 'iii15r1c1', 'iii16r1c1', 'iii16r1c2', 'iii16r1c3', 'iii17r1c1', 'iii18r1c1', 'iii19r1c1', 'iv1r1c1', 'iv2r1c1', 'iv3r1c1', 'iv4r1c1', 'iv4r2c1', 'iv4r3c1', 'iv4

**Interpretación.** Los nombres son códigos de pregunta (`iv12r13c1` = sección IV, pregunta 12, fila 13, columna 1). Están en **minúscula** (excepto `FEX_c` y algunos identificadores), mientras que el diccionario oficial los escribe en **mayúscula** (`IV12R13C1`). Para relacionarlos hay que comparar sin distinguir mayúsculas.

In [30]:
if df_dane is not None:
    display(df_dane.dtypes)

ID                int64
UID_ENCUESTA        str
uid_padre           str
UUID                str
UUID_1              str
                 ...   
v7r2c1          float64
v7r3c1          float64
v7r4c1          float64
v7r5c1          float64
v7r6c1          float64
Length: 377, dtype: object

In [31]:
if df_dane is not None:
    display(df_dane.dtypes.value_counts())

float64    347
int64       25
str          5
Name: count, dtype: int64

**Interpretación.** Hay **347 columnas `float64`, 25 `int64` y 5 de texto**. Las de texto son cuatro identificadores y `FEX_c`. Que `FEX_c` sea texto es un problema: es un **factor de expansión** (un número) y, por estar escrito con coma decimal, pandas no lo reconoce como número. Las columnas `float64` son, en su mayoría, códigos de respuesta (por ejemplo, `1 = Sí`, `2 = No`).

In [32]:
if df_dane is not None:
    df_dane.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 148700 entries, 0 to 148699
Data columns (total 377 columns):
 #    Column        Non-Null Count   Dtype  
---   ------        --------------   -----  
 0    ID            148700 non-null  int64  
 1    UID_ENCUESTA  148700 non-null  str    
 2    uid_padre     148700 non-null  str    
 3    UUID          148700 non-null  str    
 4    UUID_1        148700 non-null  str    
 5    p1r1c1        148700 non-null  int64  
 6    p2r1c1        148700 non-null  int64  
 7    p3r1c1        148700 non-null  int64  
 8    FEX_c         148700 non-null  str    
 9    ii1r1c1       148700 non-null  int64  
 10   ii1r1c2       144655 non-null  float64
 11   ii2r1c1       148700 non-null  int64  
 12   ii3r1c1       148700 non-null  int64  
 13   ii4r1c1       148700 non-null  int64  
 14   ii5r1c1       148700 non-null  int64  
 15   ii7r1c1       148700 non-null  int64  
 16   ii6r1c2       40015 non-null   float64
 17   ii7r2c1       148700 non-null  int64  

**Interpretación.** Solo una parte de las columnas está completa en las 148.700 filas; la mayoría tiene faltantes y una (`iv13r20c2`) no tiene ningún valor. En una encuesta de hogares esto suele deberse a **preguntas que solo se hacen a algunas personas** (por ejemplo, las de redes sociales solo a quien usa internet para redes). El conteo exacto y su explicación se revisan en la sección 5.

In [33]:
if df_dane is not None:
    display(df_dane.describe())

,ID,p1r1c1,p2r1c1,p3r1c1,ii1r1c1,ii1r1c2,ii2r1c1,ii3r1c1,ii4r1c1,ii5r1c1,ii7r1c1,ii6r1c2,ii7r2c1,ii6r2c2,ii7r3c1,...,v6r2c1,v6r3c1,v6r4c1,v6r5c1,v6r6c1,v6r7c1,v6r8c1,v6r9c1,v6r10c1,v7r1c1,v7r2c1,v7r3c1,v7r4c1,v7r5c1,v7r6c1
count,148700.000000,148700.000000,148700.000000,148700.000000,148700.000000,144655.000000,148700.000000,148700.000000,148700.000000,148700.000000,148700.000000,40015.000000,148700.000000,89032.000000,148700.000000,...,99064.000000,99064.000000,99064.000000,99064.000000,99064.000000,99064.000000,99064.000000,99064.000000,99064.000000,6964.000000,6964.000000,4051.000000,6964.000000,3051.000000,3591.000000
mean,87446.210377,44.774264,45062.669274,1.612495,1.027202,2.028654,1.010282,3.562703,2.565293,1.918978,1.730901,1.077321,1.401264,1.300297,1.969153,...,2.019018,2.005411,2.002776,2.002140,2.007904,2.021309,2.002837,2.009024,2.013769,1.483199,1.747128,1.681560,1.483199,1.355293,1.797549
std,16417.993931,29.673332,29660.203245,0.861428,0.162673,1.771237,0.123153,1.684965,1.037962,0.272870,0.443493,0.363625,0.490156,0.681809,0.172905,...,0.168006,0.195213,0.183130,0.180669,0.155482,0.176385,0.189469,0.148479,0.133564,0.499754,0.434689,0.465929,0.499754,0.478681,0.401882
min,59694.000000,5.000000,5001.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,73369.750000,17.000000,17867.000000,1.000000,1.000000,1.000000,1.000000,2.000000,2.000000,2.000000,1.000000,1.000000,1.000000,1.000000,2.000000,...,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000
50%,86773.000000,44.000000,44430.000000,1.000000,1.000000,2.000000,1.000000,3.000000,2.000000,2.000000,2.000000,1.000000,1.000000,1.000000,2.000000,...,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.000000,2.000000,2.000000,1.000000,1.000000,2.000000
75%,101060.000000,70.000000,70670.000000,3.000000,1.000000,2.000000,1.000000,4.000000,3.000000,2.000000,2.000000,1.000000,2.000000,1.000000,2.000000,...,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
max,119604.000000,99.000000,99773.000000,3.000000,2.000000,9.000000,4.000000,16.000000,7.000000,2.000000,2.000000,16.000000,2.000000,38.000000,2.000000,...,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000


**Interpretación.** `describe()` del DANE resume 372 columnas numéricas, casi todas códigos. Por eso se revisan aparte las variables pertinentes para el proyecto (siguiente celda).

In [34]:
if df_dane is not None:
    VARIABLES_DANE = {
        "p1r1c1": "Departamento (código)",
        "p2r1c1": "Municipio (código)",
        "iii4r1c1": "Sexo al nacer",
        "iii7r1c1": "Años cumplidos",
        "iv5r1c1": "Frecuencia de uso de internet",
        "iv10r5c1": "¿Usa internet para redes sociales?",
        "iv12r2c1": "Redes para ver contenidos de otros y comentar",
        "iv12r13c1": "Redes para entretenimiento (seguir influenciadores)",
    }
    df_dane_sel = df_dane[list(VARIABLES_DANE)]
    display(df_dane_sel.describe())

,p1r1c1,p2r1c1,iii4r1c1,iii7r1c1,iv5r1c1,iv10r5c1,iv12r2c1,iv12r13c1
count,148700.000000,148700.000000,148700.000000,148700.000000,139223.000000,99064.000000,80840.000000,80840.000000
mean,44.774264,45062.669274,1.522024,35.626873,2.368423,1.183962,1.467900,1.224171
std,29.673332,29660.203245,0.499516,22.313871,1.730378,0.387455,0.498972,0.417038
min,5.000000,5001.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,17.000000,17867.000000,1.000000,16.000000,1.000000,1.000000,1.000000,1.000000
50%,44.000000,44430.000000,2.000000,34.000000,1.000000,1.000000,1.000000,1.000000
75%,70.000000,70670.000000,2.000000,53.000000,5.000000,1.000000,2.000000,1.000000
max,99.000000,99773.000000,2.000000,107.000000,5.000000,2.000000,2.000000,2.000000


**Interpretación.**
- `iii7r1c1` (edad): va de **0 a 107 años**. El DANE encuesta a todas las personas del hogar, no solo a jóvenes; para el proyecto habría que filtrar 18 a 25.
- `iii4r1c1` va de 1 a 2 (Hombre, Mujer), coherente con el diccionario.
- `iv10r5c1`, `iv12r2c1` e `iv12r13c1` van de 1 a 2 (Sí, No) y tienen `count` menores que 148.700: confirman preguntas condicionadas.
- `p2r1c1` (municipio) tiene como mínimo `5001`: es el código de Medellín (05001) **sin el cero inicial**, porque se guardó como número.

In [35]:
if df_dane is not None:
    columnas_texto = df_dane.select_dtypes(include="object").columns
    display(df_dane[columnas_texto].describe(include="object"))

C:\Users\maile\AppData\Local\Temp\ipykernel_27604\2717524219.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = df_dane.select_dtypes(include="object").columns
C:\Users\maile\AppData\Local\Temp\ipykernel_27604\2717524219.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/use

,UID_ENCUESTA,uid_padre,UUID,UUID_1,FEX_c
count,148700,148700,148700,148700,148700
unique,53343,53343,53570,148700,38842
top,9b1b1ce8-a8cc-4e98-bd52-1631f9db5270,9b1b1ce8-a8cc-4e98-bd52-1631f9db5270,9b1b1ce8-a8cc-4e98-bd52-1631f9db5270/Hogares[1],a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1]/Personas[1],"32,93505021"
freq,16,16,16,1,151


**Interpretación.**
- `UUID_1` tiene **148.700 valores únicos**, uno por fila: es el identificador de cada persona (termina en `/Personas[n]`).
- `UID_ENCUESTA` tiene 53.343 valores únicos y `UUID` (que termina en `/Hogares[n]`) tiene 53.570. Esto confirma que varias personas comparten encuesta y hogar.
- `FEX_c` tiene 38.842 valores distintos y el más frecuente (`32,93505021`) se repite 151 veces. Todos sus valores son texto con coma decimal.

El aviso `Pandas4Warning` es el mismo que se explicó en Pritchard.

## 4. Tabla de diagnóstico estructural

La tabla se construye con los resultados de `shape` y `dtypes` de cada DataFrame, para que las cifras salgan directamente de los datos.

In [36]:
def resumen_estructural(nombre, df):
    tipos = ", ".join(f"{tipo}: {n}" for tipo, n in df.dtypes.astype(str).value_counts().items())
    return {"Dataset": nombre, "Filas": df.shape[0], "Columnas": df.shape[1], "Tipos de datos": tipos}


resumenes = [resumen_estructural("Pritchard (2021)", df_pritchard),
             resumen_estructural("Engeln et al. (2020)", df_engeln)]
if df_dane is not None:
    resumenes.append(resumen_estructural("DANE ENTIC Hogares 2024", df_dane))

tabla_estructural = pd.DataFrame(resumenes)
(RAIZ / "reports").mkdir(exist_ok=True)
tabla_estructural.to_csv(RAIZ / "reports" / "tabla_estructural.csv", index=False)
tabla_estructural

,Dataset,Filas,Columnas,Tipos de datos
0,Pritchard (2021),404,246,"float64: 221, str: 25"
1,Engeln et al. (2020),308,70,float64: 70
2,DANE ENTIC Hogares 2024,148700,377,"float64: 347, int64: 25, str: 5"


| Dataset | Observaciones |
|---|---|
| Pritchard (2021) | Cada fila es una persona encuestada. Las respuestas cerradas son **códigos guardados como `float64`**; necesitan el diccionario para interpretarse. Las respuestas abiertas vacías están como texto `""` y `info()` no las cuenta como faltantes. Hay edades de 17 y 26 años y bloques de preguntas con muchas menos respuestas (Twitter, Snapchat). |
| Engeln et al. (2020) | Cada fila es una participante del experimento, ordenada por grupo (`Condition`). Todo es `float64`; no tiene texto. Las variables de la tarea tienen unos 106 faltantes, probablemente del grupo de control. Hay al menos una edad de 26 años. |
| DANE ENTIC Hogares 2024 | Cada fila es una persona; las filas no son hogares (53.343 encuestas). Nombres de columna en código y en minúscula, distintos del diccionario. `FEX_c` es texto con coma decimal. Muchas columnas con faltantes por preguntas condicionadas y edades de 0 a 107 años. Solo aporta contexto de exposición. |

En la sección 5 se cuantifican estos puntos y se proponen las acciones de limpieza.